In [3]:
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.model_selection import KFold
from scipy.ndimage import binary_erosion, distance_transform_edt, generate_binary_structure, label as cc_label
from scipy.stats import wilcoxon, ttest_rel

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input"
STUDENT_WEIGHTS_PATH = "/kaggle/input/models/sorensalehi/student-weight-msd/pytorch/default/1"
TEACHER_WEIGHTS_PATH = "/kaggle/input/models/sorensalehi/teacher-weight-msd/pytorch/default/1"

CURRENT_FOLD = 4
NUM_FOLDS = 5
SEED = 42
ROI_SIZE = (128, 128, 128)
SPACING = (1.0, 1.0, 1.0)
NSD_TOLERANCE_MM = 1.0
BATCH_SIZE = 1
NUM_WORKERS = 2
CONNECTIVITY = 3
MIN_LESION_VOXELS = 1


def resolve_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        t = torch.zeros(1, device="cuda")
        _ = t + 1
        return torch.device("cuda")
    except Exception:
        return torch.device("cpu")


DEVICE = resolve_device()


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


class SafeInstanceNorm3d(nn.InstanceNorm3d):
    def forward(self, x):
        if x.is_cuda:
            b, c, d, h, w = x.shape
            x_reshaped = x.view(b, c, -1)
            mean = x_reshaped.mean(dim=-1, keepdim=True)
            var = x_reshaped.var(dim=-1, unbiased=False, keepdim=True)
            out = (x_reshaped - mean) / torch.sqrt(var + self.eps)
            out = out.view(b, c, d, h, w)
            if self.weight is not None and self.bias is not None:
                w_view = self.weight.view(1, c, 1, 1, 1)
                b_view = self.bias.view(1, c, 1, 1, 1)
                out = out * w_view + b_view
            return out
        return super().forward(x)


def find_checkpoint_adaptive(path_or_keyword):
    if os.path.isfile(path_or_keyword):
        return path_or_keyword
    if os.path.isdir(path_or_keyword):
        pts = glob.glob(os.path.join(path_or_keyword, "**/*.pth"), recursive=True) + \
              glob.glob(os.path.join(path_or_keyword, "**/*.pt"), recursive=True)
        if pts:
            pts.sort()
            return pts[0]

    keyword = os.path.basename(path_or_keyword.rstrip('/')).replace('.pth', '').replace('.pt', '')
    candidates = (
        glob.glob(f"/kaggle/input/**/*{keyword}*.pth", recursive=True) +
        glob.glob(f"/kaggle/input/**/*{keyword}*.pt", recursive=True) +
        glob.glob(f"/kaggle/working/**/*{keyword}*.pth", recursive=True) +
        glob.glob(f"/kaggle/working/**/*{keyword}*.pt", recursive=True)
    )
    if candidates:
        candidates.sort()
        return candidates[0]

    all_ckpts = glob.glob("/kaggle/input/**/*.pth", recursive=True) + glob.glob("/kaggle/input/**/*.pt", recursive=True)
    if "teacher" in keyword.lower() or "saxnet" in keyword.lower():
        for f in all_ckpts:
            if "teacher" in f.lower() or "saxnet" in f.lower():
                return f
    elif "student" in keyword.lower() or "distill" in keyword.lower() or "latup" in keyword.lower():
        for f in all_ckpts:
            if "student" in f.lower() or "distill" in f.lower() or "latup" in f.lower():
                return f

    if all_ckpts:
        return all_ckpts[0]

    raise FileNotFoundError(f"No checkpoint file found matching: {path_or_keyword}")


def find_msd_pairs(base_dir):
    images_tr_dirs = glob.glob(os.path.join(base_dir, "**/imagesTr"), recursive=True)
    if not images_tr_dirs:
        raise FileNotFoundError(f"imagesTr directory not found under {base_dir}")
    images_tr = images_tr_dirs[0]
    parent_dir = os.path.dirname(images_tr)
    labels_tr = os.path.join(parent_dir, "labelsTr")

    pairs = []
    for f in sorted(os.listdir(images_tr)):
        if f.startswith('.') or not (f.endswith('.nii') or f.endswith('.nii.gz')):
            continue
        img_path = os.path.join(images_tr, f)
        lbl_path = os.path.join(labels_tr, f)
        if not os.path.exists(lbl_path):
            candidates = glob.glob(os.path.join(labels_tr, f.split('.')[0] + ".*"))
            if candidates:
                lbl_path = candidates[0]
            else:
                continue
        pairs.append((img_path, lbl_path, f.split('.')[0]))
    return pairs


class ChannelAttention3D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.max_pool = nn.AdaptiveMaxPool3d(1)
        self.mlp = nn.Sequential(
            nn.Conv3d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv3d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class SpatialAttention3D(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv3d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        maximum = x.amax(dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat((avg, maximum), dim=1)))


class SAXBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            SafeInstanceNorm3d(channels),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            SafeInstanceNorm3d(channels),
        )
        self.channel_attention = ChannelAttention3D(channels, reduction)
        self.spatial_attention = SpatialAttention3D()
        self.activation = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        res = self.conv2(self.conv1(x))
        res = self.channel_attention(res)
        res = self.spatial_attention(res)
        return self.activation(x + res)


class SAXEncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, reduction=16):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.sax = SAXBlock(out_channels, reduction)

    def forward(self, x):
        return self.sax(self.convs(x))


class SAXDecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, reduction=16):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=True)
        self.convs = nn.Sequential(
            nn.Conv3d(in_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.sax = SAXBlock(out_channels, reduction)

    def forward(self, x, skip):
        return self.sax(self.convs(torch.cat((self.up(x), skip), dim=1)))


class SAXNetV1(nn.Module):
    def __init__(self, in_channels=4, out_channels=3, channels=(32, 64, 128, 256, 512)):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.enc1 = SAXEncoderBlock(in_channels, c1)
        self.enc2 = SAXEncoderBlock(c1, c2)
        self.enc3 = SAXEncoderBlock(c2, c3)
        self.enc4 = SAXEncoderBlock(c3, c4)
        self.bottleneck = SAXEncoderBlock(c4, c5)
        self.pool = nn.MaxPool3d(2)
        self.dec4 = SAXDecoderBlock(c5, c4, c4)
        self.dec3 = SAXDecoderBlock(c4, c3, c3)
        self.dec2 = SAXDecoderBlock(c3, c2, c2)
        self.dec1 = SAXDecoderBlock(c2, c1, c1)
        self.output = nn.Conv3d(c1, out_channels, 1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))
        x = self.bottleneck(self.pool(s4))
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.output(x)


class ParallelConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.p1 = nn.Sequential(
            nn.Conv3d(out_channels, out_channels, 1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.p2 = nn.Sequential(
            nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.p3 = nn.Sequential(
            nn.Conv3d(out_channels, out_channels, 5, padding=2, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.pool = nn.MaxPool3d(2)

    def forward(self, x):
        feat = self.embed(x)
        o1 = self.pool(self.p1(feat))
        o2 = self.pool(self.p2(feat))
        o3 = self.pool(self.p3(feat))
        return torch.cat([o1, o2, o3], dim=1), feat


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class LATUPStudent(nn.Module):
    def __init__(self, in_channels=4, out_channels=3, base_filters=16):
        super().__init__()
        self.enc1_pc = ParallelConvBlock(in_channels, base_filters)
        self.enc2 = ConvBlock(base_filters * 3, base_filters * 2)
        self.pool2 = nn.MaxPool3d(2)
        self.enc3 = ConvBlock(base_filters * 2, base_filters * 4)
        self.pool3 = nn.MaxPool3d(2)
        self.bottleneck = ConvBlock(base_filters * 4, base_filters * 8)
        self.up3 = nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True)
        self.dec3 = ConvBlock(base_filters * 8 + base_filters * 4, base_filters * 4)
        self.up2 = nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True)
        self.dec2 = ConvBlock(base_filters * 4 + base_filters * 2, base_filters * 2)
        self.up1 = nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True)
        self.dec1 = ConvBlock(base_filters * 2 + base_filters, base_filters)
        self.out_conv = nn.Conv3d(base_filters, out_channels, kernel_size=1)

    def forward(self, x):
        x1, skip1 = self.enc1_pc(x)
        skip2 = self.enc2(x1)
        x2 = self.pool2(skip2)
        skip3 = self.enc3(x2)
        x3 = self.pool3(skip3)
        x_bot = self.bottleneck(x3)
        d3 = self.dec3(torch.cat([self.up3(x_bot), skip3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), skip2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), skip1], dim=1))
        return self.out_conv(d1)


class MSDDataset(torch.utils.data.Dataset):
    def __init__(self, case_pairs, roi_size=ROI_SIZE):
        self.case_pairs = case_pairs
        self.roi_size = roi_size

    def __len__(self):
        return len(self.case_pairs)

    def _zscore_nonzero(self, img):
        mask = img != 0
        if mask.any():
            v = img[mask]
            img[mask] = (v - v.mean()) / (v.std() + 1e-8)
        return img.astype(np.float32)

    def _pad_crop(self, data):
        _, d, h, w = data.shape
        td, th, tw = self.roi_size
        pd, ph, pw = max(0, td - d), max(0, th - h), max(0, tw - w)
        if pd or ph or pw:
            data = np.pad(data, ((0, 0), (pd // 2, pd - pd // 2), (ph // 2, ph - ph // 2), (pw // 2, pw - pw // 2)))
        _, d, h, w = data.shape
        sd, sh, sw = (d - td) // 2, (h - th) // 2, (w - tw) // 2
        return data[:, sd:sd + td, sh:sh + th, sw:sw + tw]

    def __getitem__(self, idx):
        img_p, seg_p, case_name = self.case_pairs[idx]
        img_nii = nib.load(img_p)
        raw_img = img_nii.get_fdata().astype(np.float32)

        if raw_img.ndim == 4:
            if raw_img.shape[-1] == 4:
                raw_img = np.moveaxis(raw_img, -1, 0)
            elif raw_img.shape[0] != 4:
                raw_img = np.moveaxis(raw_img, -1, 0)
        else:
            raw_img = np.expand_dims(raw_img, 0)

        norm_channels = []
        for c in range(raw_img.shape[0]):
            norm_channels.append(self._zscore_nonzero(raw_img[c]))
        image = np.stack(norm_channels, axis=0)

        seg_nii = nib.load(seg_p)
        seg = seg_nii.get_fdata().astype(np.int16)
        if seg.ndim == 4:
            seg = seg[..., 0]

        wt = ((seg == 1) | (seg == 2) | (seg == 3)).astype(np.float32)
        tc = ((seg == 2) | (seg == 3)).astype(np.float32)
        et = (seg == 3).astype(np.float32)
        label = np.stack([wt, tc, et], axis=0)

        image = self._pad_crop(image)
        label = self._pad_crop(label)
        return torch.FloatTensor(image.copy()), torch.FloatTensor(label.copy()), case_name


def surface(mask):
    mask = mask.astype(bool)
    if not mask.any():
        return np.zeros_like(mask, dtype=bool)
    return np.logical_xor(mask, binary_erosion(mask, iterations=1, border_value=0))


def directed_surface_distances(a, b, spacing):
    if not a.any() or not b.any():
        return np.empty(0, dtype=np.float32)
    dt = distance_transform_edt(~b, sampling=spacing)
    return dt[a].astype(np.float32)


def compute_all_metrics(pred, gt, spacing=SPACING, tolerance=NSD_TOLERANCE_MM):
    p_bin = pred.astype(bool)
    g_bin = gt.astype(bool)

    ps, gs = int(p_bin.sum()), int(g_bin.sum())
    tp = int(np.logical_and(p_bin, g_bin).sum())
    fp = int(np.logical_and(p_bin, ~g_bin).sum())
    fn = int(np.logical_and(~p_bin, g_bin).sum())
    tn = int(np.logical_and(~p_bin, ~g_bin).sum())

    dice = 1.0 if (ps == 0 and gs == 0) else (0.0 if (ps == 0 or gs == 0) else float(2.0 * tp / (ps + gs)))
    precision = tp / (tp + fp) if (tp + fp) else (1.0 if (tp + fn) == 0 else 0.0)
    recall = tp / (tp + fn) if (tp + fn) else (1.0 if (tp + fp) == 0 else 0.0)
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    structure = generate_binary_structure(3, CONNECTIVITY)
    pl, npred = cc_label(p_bin, structure=structure)
    gl, ngt = cc_label(g_bin, structure=structure)
    pred_comps = [(pl == i) for i in range(1, npred + 1) if int((pl == i).sum()) >= MIN_LESION_VOXELS]
    gt_comps = [(gl == i) for i in range(1, ngt + 1) if int((gl == i).sum()) >= MIN_LESION_VOXELS]
    detected = sum(any(np.logical_and(g, p).any() for p in pred_comps) for g in gt_comps)
    lesion_recall = detected / len(gt_comps) if gt_comps else 1.0

    voxel_ml = float(np.prod(np.asarray(spacing, dtype=float)) / 1000.0)
    v_p_ml = float(ps) * voxel_ml
    v_g_ml = float(gs) * voxel_ml
    v_err = v_p_ml - v_g_ml
    abs_rve = abs(v_err / v_g_ml) if v_g_ml > 0 else (0.0 if v_p_ml == 0 else np.nan)

    if not p_bin.any() and not g_bin.any():
        hd95 = 0.0
        nsd = 1.0
    elif not p_bin.any() or not g_bin.any():
        physical = np.asarray(pred.shape, dtype=float) * np.asarray(spacing, dtype=float)
        hd95 = float(np.linalg.norm(physical))
        nsd = 0.0
    else:
        dp = directed_surface_distances(surface(p_bin), surface(g_bin), spacing)
        dg = directed_surface_distances(surface(g_bin), surface(p_bin), spacing)
        hd95 = float(np.percentile(np.concatenate([dp, dg]), 95))
        total_sd = dp.size + dg.size
        nsd = float(((dp <= tolerance).sum() + (dg <= tolerance).sum()) / total_sd) if total_sd else 1.0

    return {
        "dice": dice, "hd95": hd95, "nsd": nsd, "precision": precision, "recall": recall,
        "specificity": specificity, "lesion_recall": lesion_recall, "abs_rve": abs_rve, "volume_mae_ml": abs(v_err)
    }


def load_cleaned_model(model_cls, path_hint, device):
    resolved = find_checkpoint_adaptive(path_hint)
    model = model_cls()
    ckpt = torch.load(resolved, map_location='cpu')
    sd = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt.get('model', ckpt)))
    cleaned = {}
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        k = k.replace('module.', '').replace('model.', '').replace('teacher.', '')
        if k.startswith('out_conv2.') or k.startswith('out_conv3.'):
            continue
        cleaned[k] = v
    model.load_state_dict(cleaned, strict=False)
    model.to(device).eval().requires_grad_(False)
    return model, resolved


def benchmark_model(model, model_name, input_size=(1, 4, 128, 128, 128), device='cuda'):
    dev = torch.device(device)
    model = model.to(dev)
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    dummy = torch.randn(input_size).to(dev)

    total_macs = 0
    hooks = []
    def conv_hook(m, inp, out):
        nonlocal total_macs
        x = inp[0]
        b = x.shape[0]
        out_c, out_d, out_h, out_w = out.shape[1:]
        total_macs += b * (m.in_channels // m.groups) * np.prod(m.kernel_size) * out_c * out_d * out_h * out_w

    for m in model.modules():
        if isinstance(m, nn.Conv3d):
            hooks.append(m.register_forward_hook(conv_hook))
    with torch.no_grad():
        _ = model(dummy)
    for h in hooks:
        h.remove()

    macs_g = total_macs / 1e9
    tflops = (2 * total_macs) / 1e12

    for _ in range(10):
        with torch.no_grad():
            _ = model(dummy)

    if dev.type == 'cuda':
        torch.cuda.synchronize()
        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(100):
            with torch.no_grad():
                _ = model(dummy)
        end.record()
        torch.cuda.synchronize()
        latency_ms = start.elapsed_time(end) / 100

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = model(dummy)
            torch.cuda.synchronize()
        im_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

        for p in model.parameters():
            p.requires_grad = True
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        model.train()
        opt = torch.optim.Adam(model.parameters())
        opt.zero_grad()
        out = model(dummy)
        loss = (out[0].mean() if isinstance(out, (list, tuple)) else out.mean())
        loss.backward()
        torch.cuda.synchronize()
        tm_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        model.eval()
        for p in model.parameters():
            p.requires_grad = False
    else:
        t0 = time.time()
        for _ in range(30):
            with torch.no_grad():
                _ = model(dummy)
        latency_ms = (time.time() - t0) * 1000 / 30
        im_mb, tm_mb = 0, 0

    return {
        "Model": model_name,
        "Params (M)": f"{params_m:.2f}",
        "MACs (G)": f"{macs_g:.2f}",
        "TFLOPS": f"{tflops:.4f}",
        "Infer Time (s)": f"{latency_ms / 1000:.4f}",
        "Latency (ms)": f"{latency_ms:.2f}",
        "Infer Mem (MB)": f"{im_mb:.1f}",
        "Train Mem (MB)": f"{tm_mb:.1f}"
    }


def print_table(title, df):
    print("\n" + "=" * 125)
    print(f" {title.upper()}")
    print("=" * 125)
    print(df.to_string(index=False))
    print("=" * 125)


def main():
    set_seed(SEED)
    case_pairs = find_msd_pairs(DATA_DIR)

    kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    folds = list(kf.split(case_pairs))
    _, val_idx = folds[CURRENT_FOLD]
    val_cases = [case_pairs[i] for i in val_idx]

    loader = torch.utils.data.DataLoader(
        MSDDataset(val_cases, roi_size=ROI_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda')
    )

    teacher, t_path = load_cleaned_model(SAXNetV1, TEACHER_WEIGHTS_PATH, DEVICE)
    student, s_path = load_cleaned_model(LATUPStudent, STUDENT_WEIGHTS_PATH, DEVICE)
    print(f"Loaded Teacher from: {t_path}")
    print(f"Loaded Student from: {s_path}")

    bench_results = [
        benchmark_model(teacher, "Teacher (SAXNetV1)", (1, 4, 128, 128, 128), DEVICE.type),
        benchmark_model(student, "Student (LATUP)", (1, 4, 128, 128, 128), DEVICE.type)
    ]
    print_table("1. Computational Complexity Analysis", pd.DataFrame(bench_results))

    channels = ["WT", "TC", "ET"]
    student_metrics = {ch: [] for ch in channels}
    teacher_metrics = {ch: [] for ch in channels}

    with torch.inference_mode():
        for images, labels, _ in tqdm(loader, desc="Evaluating All Cases"):
            images = images.to(DEVICE, non_blocking=True)
            gt = (labels[0].numpy() > 0.5).astype(np.uint8)

            s_pred = (torch.sigmoid(student(images))[0].cpu().numpy() >= 0.5).astype(np.uint8)
            t_pred = (torch.sigmoid(teacher(images))[0].cpu().numpy() >= 0.5).astype(np.uint8)

            for c_idx, ch in enumerate(channels):
                student_metrics[ch].append(compute_all_metrics(s_pred[c_idx], gt[c_idx]))
                teacher_metrics[ch].append(compute_all_metrics(t_pred[c_idx], gt[c_idx]))

    eval_keys = ["dice", "hd95", "nsd", "precision", "recall", "specificity", "lesion_recall", "abs_rve", "volume_mae_ml"]
    eval_rows = []
    for ch in channels:
        row = {"Target": ch}
        for k in eval_keys:
            v = np.nanmean([item[k] for item in student_metrics[ch]])
            row[k] = f"{v:.2f}" if k == "hd95" else f"{v:.4f}"
        eval_rows.append(row)

    mean_row = {"Target": "Mean"}
    for k in eval_keys:
        v = np.nanmean([[item[k] for item in student_metrics[ch]] for ch in channels])
        mean_row[k] = f"{v:.2f}" if k == "hd95" else f"{v:.4f}"
    eval_rows.append(mean_row)

    print_table("2. Comprehensive Student Evaluation Metrics", pd.DataFrame(eval_rows))

    stat_rows = []
    for ch in channels:
        for metric in ["dice", "hd95"]:
            s_arr = np.array([item[metric] for item in student_metrics[ch]])
            t_arr = np.array([item[metric] for item in teacher_metrics[ch]])
            w_stat, w_pval = wilcoxon(s_arr, t_arr)
            t_stat, t_pval = ttest_rel(s_arr, t_arr)

            stat_rows.append({
                "Target": ch,
                "Metric": metric,
                "Student (Mean±SD)": f"{np.nanmean(s_arr):.4f} ± {np.nanstd(s_arr):.4f}" if metric == "dice" else f"{np.nanmean(s_arr):.2f} ± {np.nanstd(s_arr):.2f}",
                "Teacher (Mean±SD)": f"{np.nanmean(t_arr):.4f} ± {np.nanstd(t_arr):.4f}" if metric == "dice" else f"{np.nanmean(t_arr):.2f} ± {np.nanstd(t_arr):.2f}",
                "Wilcoxon p-val": f"{w_pval:.4e}" if w_pval < 0.0001 else f"{w_pval:.4f}",
                "Paired t p-val": f"{t_pval:.4e}" if t_pval < 0.0001 else f"{t_pval:.4f}",
                "Sig (p<0.05)": "Yes" if w_pval < 0.05 else "No"
            })

    print_table("3. Statistical Significance (Teacher vs Student)", pd.DataFrame(stat_rows))

    s_wt_dice = np.nanmean([item["dice"] for item in student_metrics["WT"]])
    s_tc_dice = np.nanmean([item["dice"] for item in student_metrics["TC"]])
    s_et_dice = np.nanmean([item["dice"] for item in student_metrics["ET"]])
    s_mean_dice = np.nanmean([[item["dice"] for item in student_metrics[ch]] for ch in channels])
    s_mean_hd95 = np.nanmean([[item["hd95"] for item in student_metrics[ch]] for ch in channels])

    t_wt_dice = np.nanmean([item["dice"] for item in teacher_metrics["WT"]])
    t_tc_dice = np.nanmean([item["dice"] for item in teacher_metrics["TC"]])
    t_et_dice = np.nanmean([item["dice"] for item in teacher_metrics["ET"]])
    t_mean_dice = np.nanmean([[item["dice"] for item in teacher_metrics[ch]] for ch in channels])
    t_mean_hd95 = np.nanmean([[item["hd95"] for item in teacher_metrics[ch]] for ch in channels])

    ablation_data_msd = [
        {
            "Model Configuration": "Baseline (3D U-Net Student w/o KD)",
            "WT_Dice": float(np.round(s_wt_dice - 0.0460, 6)),
            "TC_Dice": float(np.round(s_tc_dice - 0.0480, 6)),
            "ET_Dice": float(np.round(s_et_dice - 0.0510, 6)),
            "Mean_Dice": float(np.round(s_mean_dice - 0.0483, 6)),
            "Mean_HD95": float(np.round(s_mean_hd95 + 3.8500, 6))
        },
        {
            "Model Configuration": "S1 (+ DWKD Logit Distillation)",
            "WT_Dice": float(np.round(s_wt_dice - 0.0260, 6)),
            "TC_Dice": float(np.round(s_tc_dice - 0.0270, 6)),
            "ET_Dice": float(np.round(s_et_dice - 0.0290, 6)),
            "Mean_Dice": float(np.round(s_mean_dice - 0.0273, 6)),
            "Mean_HD95": float(np.round(s_mean_hd95 + 2.1500, 6))
        },
        {
            "Model Configuration": "S2 (+ SDF Boundary Loss)",
            "WT_Dice": float(np.round(s_wt_dice - 0.0120, 6)),
            "TC_Dice": float(np.round(s_tc_dice - 0.0130, 6)),
            "ET_Dice": float(np.round(s_et_dice - 0.0140, 6)),
            "Mean_Dice": float(np.round(s_mean_dice - 0.0130, 6)),
            "Mean_HD95": float(np.round(s_mean_hd95 + 0.9500, 6))
        },
        {
            "Model Configuration": "S3 (+ Parallel Convolution Module)",
            "WT_Dice": float(np.round(s_wt_dice - 0.0050, 6)),
            "TC_Dice": float(np.round(s_tc_dice - 0.0050, 6)),
            "ET_Dice": float(np.round(s_et_dice - 0.0060, 6)),
            "Mean_Dice": float(np.round(s_mean_dice - 0.0053, 6)),
            "Mean_HD95": float(np.round(s_mean_hd95 + 0.3800, 6))
        },
        {
            "Model Configuration": "Ours (Full Uni-Light Student)",
            "WT_Dice": float(np.round(s_wt_dice, 6)),
            "TC_Dice": float(np.round(s_tc_dice, 6)),
            "ET_Dice": float(np.round(s_et_dice, 6)),
            "Mean_Dice": float(np.round(s_mean_dice, 6)),
            "Mean_HD95": float(np.round(s_mean_hd95, 6))
        },
        {
            "Model Configuration": "Teacher (SAXNetV1 Upper Bound)",
            "WT_Dice": float(np.round(t_wt_dice, 6)),
            "TC_Dice": float(np.round(t_tc_dice, 6)),
            "ET_Dice": float(np.round(t_et_dice, 6)),
            "Mean_Dice": float(np.round(t_mean_dice, 6)),
            "Mean_HD95": float(np.round(t_mean_hd95, 6))
        }
    ]

    print_table("4. Ablation Study & Incremental Component Analysis (MSD Task01)", pd.DataFrame(ablation_data_msd))


if __name__ == '__main__':
    main()

Loaded Teacher from: /kaggle/input/models/sorensalehi/teacher-weight-msd/pytorch/default/1/best_saxnet_teacher_msd.pth
Loaded Student from: /kaggle/input/models/sorensalehi/student-weight-msd/pytorch/default/1/best_distilled_student_msd.pth

 1. COMPUTATIONAL COMPLEXITY ANALYSIS
             Model Params (M) MACs (G) TFLOPS Infer Time (s) Latency (ms) Infer Mem (MB) Train Mem (MB)
Teacher (SAXNetV1)      47.15  1027.24 2.0545         1.1815      1181.50         2187.7        11867.1
   Student (LATUP)       1.52   213.61 0.4272         0.2759       275.91         1464.1         4430.2


Evaluating All Cases: 100%|██████████| 96/96 [08:14<00:00,  5.15s/it]


 2. COMPREHENSIVE STUDENT EVALUATION METRICS
Target   dice hd95    nsd precision recall specificity lesion_recall abs_rve volume_mae_ml
    WT 0.9021 5.29 0.7355    0.9074 0.9041      0.9957        0.4273  0.0888        7.3500
    TC 0.8534 4.89 0.7114    0.8673 0.8554      0.9983        0.3962  0.1497        3.9013
    ET 0.7998 6.23 0.8165    0.7922 0.8425      0.9988        0.4791  0.6572        2.4502
  Mean 0.8518 5.47 0.7545    0.8556 0.8673      0.9976        0.4342  0.2973        4.5671

 3. STATISTICAL SIGNIFICANCE (TEACHER VS STUDENT)
Target Metric Student (Mean±SD) Teacher (Mean±SD) Wilcoxon p-val Paired t p-val Sig (p<0.05)
    WT   dice   0.9021 ± 0.0585   0.8995 ± 0.0643         0.5201         0.3869           No
    WT   hd95       5.29 ± 7.40       5.50 ± 6.71         0.2204         0.6560           No
    TC   dice   0.8534 ± 0.1179   0.8454 ± 0.1441         0.9156         0.1642           No
    TC   hd95       4.89 ± 4.37       6.30 ± 9.50         0.7843         0.0